# Metric Learning with Contrastive-style Losses (Optional)

This optional notebook explores **supervised metric learning**: instead of predicting a class label
directly, we train an encoder network to produce embeddings where images of the same digit end up
close together and images of different digits end up far apart. We compare three losses that all
share the same "pull positives together" idea, but differ in whether (and how) they push negatives
apart:

1. **Min-distance loss** — pulls positive pairs together, with no repulsion term (spoiler: this collapses).
2. **Triplet loss** — pulls positives together *and* pushes one negative away, with a margin.
3. **N-pair loss** — pulls positives together while pushing *several* negatives away at once, using a
   softmax-style formulation closely related to InfoNCE (the loss behind self-supervised methods like SimCLR).

**Prerequisites:** Sessions 6-7 (CNNs, training loops, embeddings/losses).

**Runtime:** all three trainings run on CPU by design — `EncoderNet` is small and the pair/triplet
mining is numpy-based (not GPU-friendly), so a device transfer buys nothing here. Epoch counts below
were chosen for class-time feasibility; the whole notebook completes in well under 15 minutes.

In [ ]:
from tqdm import tqdm
from sklearn.manifold import TSNE
import numpy as np
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from typing import Tuple, Callable
from matplotlib import pyplot as plt

# Seeds for reproducibility (weight init, mining, and t-SNE are all stochastic)
torch.manual_seed(0)
np.random.seed(0)

In [ ]:
# Utility functions
def get_embeddings(model: torch.nn.Module, loader: DataLoader) -> Tuple[torch.Tensor, torch.Tensor]:
    model.eval()
    n_samples = 300

    # Infer one batch from the test set
    with torch.no_grad():
        images, labels = next(iter(loader))
        embeddings = model(images[:n_samples])
        labels = labels[:n_samples]

    return embeddings, labels


def reduce_dimensions(embeddings: torch.Tensor) -> np.ndarray:
    tsne = TSNE(n_components=2, random_state=42)
    embeddings_2d = tsne.fit_transform(embeddings.numpy())
    return embeddings_2d


def plot_embeddings(embeddings: np.ndarray, labels: torch.Tensor):
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(embeddings[:, 0], embeddings[:, 1], c=labels, cmap="tab10")
    plt.colorbar(scatter)
    plt.title("2D t-SNE visualization of learned embeddings")
    plt.xlabel("t-SNE dimension 1")
    plt.ylabel("t-SNE dimension 2")
    plt.show()


def generate_2d_plot(model: torch.nn.Module, test_loader: DataLoader):
    embeddings, labels = get_embeddings(model, test_loader)
    embeddings_2d = reduce_dimensions(embeddings)
    plot_embeddings(embeddings_2d, labels)

In [ ]:
# Get the MNIST dataset
# Set up the data preprocessing and loading:
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
)  # transform the data to torch tensor and normalize
train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Now let's split the training set into a training and validation set
generator = torch.Generator().manual_seed(42)  # just a random generator
train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [50000, 10000], generator=generator)

print(
    "We loaded {} training, {} validation, and {} testing samples".format(
        len(train_dataset), len(val_dataset), len(test_dataset)
    )
)

# Set up the data loaders (they are iterable objects that return the data in batches)
batch_size = 2048
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Visualize some of the data
examples = enumerate(test_loader)
batch_idx, (images, labels) = next(examples)
images = images.numpy()  # convert images to numpy for display
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(images[i][0], cmap="gray")
    plt.axis("off")
    plt.title("Label: {}".format(labels[i]))

# First approach: Min Distance Loss

We start with the simplest possible idea: only pull same-class pairs together. There is
(deliberately) nothing pushing different classes apart — see what happens below.

In [ ]:
# Define a simple CNN model
class EncoderNet(torch.nn.Module):
    def __init__(self):
        super(EncoderNet, self).__init__()
        self.conv1 = torch.nn.Conv2d(1, 32, 3)
        self.conv2 = torch.nn.Conv2d(32, 64, 3)
        self.conv3 = torch.nn.Conv2d(64, 128, 3)
        self.pool = torch.nn.MaxPool2d(2, 2)
        self.fc1 = torch.nn.Linear(128, 256)
        self.fc2 = torch.nn.Linear(256, 128)  # embedding dimension

    def forward(self, x):
        x = self.pool(torch.nn.functional.relu(self.conv1(x)))
        x = self.pool(torch.nn.functional.relu(self.conv2(x)))
        x = self.pool(torch.nn.functional.relu(self.conv3(x)))
        x = x.view(-1, 128)
        x = torch.nn.functional.relu(self.fc1(x))
        x = self.fc2(x)
        return torch.nn.functional.normalize(x, p=2, dim=1)


def train_model(
    model: torch.nn.Module,
    criterion: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    mining_function: Callable,
    train_loader: DataLoader,
    num_epochs: int,
) -> None:
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        num_batches = 0

        for batch_idx, (images, labels) in tqdm(enumerate(train_loader), total=len(train_loader)):
            anchor_positives_indices, positives_indices, negatives_indices = mining_function(images, labels)
            if len(anchor_positives_indices) == 0:
                continue

            anchors = images[anchor_positives_indices]
            positives = images[positives_indices]
            negatives = images[negatives_indices]

            # Forward pass
            anchor_embeddings = model(anchors)
            positive_embeddings = model(positives)
            if negatives.ndim > 4:
                negative_embeddings = []
                for negative in negatives:
                    negative_embeddings.append(model(negative))
                negative_embeddings = torch.stack(negative_embeddings)
            else:
                negative_embeddings = model(negatives)

            # Every criterion below returns per-sample losses; relu + mean is applied uniformly here.
            loss = criterion(anchor_embeddings, positive_embeddings, negative_embeddings)
            loss = torch.mean(torch.relu(loss))

            # Backward pass and optimize
            optimizer.zero_grad()
            loss.backward()
            # Clip gradients (after backward, before step) to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

        avg_loss = total_loss / num_batches
        print(f"Epoch [{epoch+1}/{num_epochs}], Average Loss: {avg_loss:.4f}")

    print("Training completed!")

In [ ]:
class MinDistanceLoss(torch.nn.Module):
    def __init__(self):
        super(MinDistanceLoss, self).__init__()

    def forward(self, anchor, positive, negative):
        # Per-sample distances. clamp_min before sqrt prevents NaN gradients if embeddings collapse
        # (positive - anchor -> 0, so the squared distance -> 0 and d/dx sqrt(x) is undefined at 0).
        distance_positive = torch.sqrt((anchor - positive).pow(2).sum(1).clamp_min(1e-8))
        return distance_positive

In [ ]:
def get_anchor_positives(images: torch.Tensor, labels: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    labels = labels.cpu().numpy()
    
    # Create a boolean mask where each row shows which indices have the same label
    label_mask = labels.reshape(-1, 1) == labels.reshape(1, -1)
    # Zero out diagonal to exclude self-pairs
    np.fill_diagonal(label_mask, 0)
    
    # Find valid anchors (samples that have at least one positive)
    valid_anchors = np.where(label_mask.sum(axis=1) > 0)[0]
    
    # For each valid anchor, randomly select one positive
    positives = np.array([
        np.random.choice(np.where(label_mask[anchor])[0])
        for anchor in valid_anchors
    ])
    
    return (torch.LongTensor(valid_anchors), 
            torch.LongTensor(positives), 
            torch.LongTensor([]))

*Runtime note: 5 epochs (shrunk from a longer run, and enough to see the collapse below) — expect
well under a minute on CPU for this small model/batch size.*

In [ ]:
model_min_distance = EncoderNet()
criterion_min_distance = MinDistanceLoss()
optimizer_min_distance = torch.optim.Adam(model_min_distance.parameters(), lr=0.001)

train_model(
    model=model_min_distance,
    criterion=criterion_min_distance,
    optimizer=optimizer_min_distance,
    mining_function=get_anchor_positives,
    train_loader=train_loader,
    num_epochs=5,
)

In [ ]:
generate_2d_plot(model_min_distance, test_loader)

**Why did the embeddings collapse?** `MinDistanceLoss` only pulls positive pairs together — there is
no term that pushes different classes apart. The trivial global optimum is to map *every* image to
the same point (loss = 0 everywhere), which is exactly the kind of degenerate solution a well-posed
metric-learning loss must avoid. That is the point of this section: pulling positives together is
necessary but not sufficient. The Triplet and N-pair losses below add an explicit repulsive term
against negatives to fix this.

# Triplet Loss

In [ ]:
# Triplet loss function
class TripletLoss(torch.nn.Module):
    def __init__(self, margin=1.0):
        super(TripletLoss, self).__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        # Per-sample losses. clamp_min before sqrt prevents NaN gradients on collapsed embeddings.
        distance_positive = torch.sqrt((anchor - positive).pow(2).sum(1).clamp_min(1e-8))
        distance_negative = torch.sqrt((anchor - negative).pow(2).sum(1).clamp_min(1e-8))
        losses = distance_positive - distance_negative + self.margin
        return losses

In [ ]:
def get_triplets(images: torch.Tensor, labels: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Get triplets for the triplet loss, processing each element in the batch.

    Args:
        images: torch.Tensor, shape (N, 1, 28, 28)
        labels: torch.Tensor, shape (N,)

    Returns:
        Tuple of anchor, positive, and negative index tensors, each of shape (M,),
        where M <= n_triplets is the number of valid triplets found.
    """
    n_triplets = 30  # cap on anchors considered per batch, keeps mining cost bounded
    labels = labels.cpu().numpy()
    anchor_indices = []
    positive_indices = []
    negative_indices = []

    # Process each sample as a potential anchor
    for idx in range(len(labels[:n_triplets])):
        anchor_label = labels[idx]

        # Find all positives (same label as anchor, excluding self)
        positive_mask = (labels == anchor_label) & (np.arange(len(labels)) != idx)
        positive_candidates = np.where(positive_mask)[0]

        # Find all negatives (different label than anchor)
        negative_mask = labels != anchor_label
        negative_candidates = np.where(negative_mask)[0]

        # Skip if we don't have both positives and negatives
        if len(positive_candidates) == 0 or len(negative_candidates) == 0:
            continue

        # Select one positive and one negative randomly
        positive_idx = np.random.choice(positive_candidates)
        negative_idx = np.random.choice(negative_candidates)

        anchor_indices.append(idx)
        positive_indices.append(positive_idx)
        negative_indices.append(negative_idx)

    return (torch.LongTensor(anchor_indices), torch.LongTensor(positive_indices), torch.LongTensor(negative_indices))

In [ ]:
model_triplet = EncoderNet()
criterion_triplet = TripletLoss()
optimizer_triplet = torch.optim.Adam(model_triplet.parameters(), lr=0.001)

train_model(
    model=model_triplet,
    criterion=criterion_triplet,
    optimizer=optimizer_triplet,
    mining_function=get_triplets,
    train_loader=train_loader,
    num_epochs=10,
)

In [ ]:
generate_2d_plot(model_triplet, test_loader)

# N-pair Loss

N-pair loss extends the triplet idea to *multiple* negatives at once, and turns the comparison into
a classification problem: given an anchor-positive pair, pick the positive out of a pool of
candidates (1 positive + K negatives). We use `exp(-‖anchor - positive‖²)` as a similarity score
(closer embeddings → higher similarity) and normalize it with a softmax-style ratio — this is the
same shape as **InfoNCE**, the loss behind most modern self-supervised contrastive methods (e.g.
SimCLR), which typically uses a temperature-scaled dot-product similarity instead. We keep the
squared-distance form here since it plugs directly into the same embeddings/mining machinery as the
losses above.

In [ ]:
class NPairLoss(torch.nn.Module):
    def __init__(self):
        super(NPairLoss, self).__init__()

    def forward(self, anchor, positive, negatives):
        """
        Args:
            anchor: torch.Tensor, shape (N, D)
            positive: torch.Tensor, shape (N, D)
            negatives: torch.Tensor, shape (N, K, D)
        """
        # exp(-distance^2): closer pairs get a HIGHER similarity score (InfoNCE-style).
        positive_similarities = torch.exp(-(anchor - positive).pow(2).sum(1))
        anchor_expanded = anchor.unsqueeze(1).expand(-1, negatives.size(1), -1)
        negative_similarities = torch.exp(-(anchor_expanded - negatives).pow(2).sum(2))

        losses = -torch.log(positive_similarities / (negative_similarities.sum(1) + positive_similarities))
        return losses

In [ ]:
def generate_n_pairs(
    images: torch.Tensor, labels: torch.Tensor, N: int = 1
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Generate n-pairs for the n-pair loss.

    Args:
        images: torch.Tensor, shape (N, 1, 28, 28)
        labels: torch.Tensor, shape (N,)
        N: int, number of negative samples per anchor

    Returns:
        Tuple containing:
        - anchor_indices: torch.Tensor, shape (M,)
        - positive_indices: torch.Tensor, shape (M,)
        - negative_indices: torch.Tensor, shape (M, N)
        where M is the number of anchor-positive pairs
    """
    labels = labels.cpu().numpy()
    anchors_list = []
    positives_list = []
    negatives_list = []

    for label in np.unique(labels):
        # Get indices for current label
        label_mask = labels == label
        label_indices = np.where(label_mask)[0]

        # Skip if we don't have enough samples
        if len(label_indices) < 2:
            continue

        # Get indices for other labels (negative samples)
        negative_mask = labels != label
        negative_indices = np.where(negative_mask)[0]

        # Skip if we don't have enough negative samples
        if len(negative_indices) < N:
            continue

        # Generate anchor-positive pairs. Sample without replacement so anchor != positive by construction.
        n_pairs = min(len(label_indices) * (len(label_indices) - 1) // 2, 5)  # limit pairs per class
        anchor_positive_pairs = np.stack(
            [np.random.choice(label_indices, size=2, replace=False) for _ in range(n_pairs)]
        )

        for anchor_idx, positive_idx in anchor_positive_pairs:
            # Select N negative samples
            negative_samples = np.random.choice(negative_indices, size=N, replace=True)

            anchors_list.append(anchor_idx)
            positives_list.append(positive_idx)
            negatives_list.append(negative_samples)

    if not anchors_list:  # If no valid pairs were found
        return torch.tensor([]), torch.tensor([]), torch.tensor([])

    # Stack all samples
    anchors = np.stack(anchors_list)
    positives = np.stack(positives_list)
    negatives = np.stack(negatives_list)

    return torch.LongTensor(anchors), torch.LongTensor(positives), torch.LongTensor(negatives)

*Runtime note: 10 epochs (reduced from the original 100 — chosen for class-time feasibility, still
enough for the N-pair loss to converge visibly) — expect a few minutes on CPU.*

In [ ]:
model_npair = EncoderNet()
criterion_npair = NPairLoss()
optimizer_npair = torch.optim.Adam(model_npair.parameters(), lr=0.001)

train_model(
    model=model_npair,
    criterion=criterion_npair,
    optimizer=optimizer_npair,
    mining_function=generate_n_pairs,
    train_loader=train_loader,
    num_epochs=10,
)

In [ ]:
generate_2d_plot(model_npair, test_loader)

## From supervised metric learning to self-supervised contrastive learning

All three losses above need **labels** to form positive/negative pairs. Self-supervised contrastive
methods (e.g. **SimCLR**) replace labels with **data augmentation**: two augmented views of the
*same* image form the positive pair, and other images in the batch (or a memory bank) serve as
negatives — no annotation required. They also typically use a **temperature-scaled dot-product
similarity** (`sim(u, v) / τ`) instead of a squared-distance-based one, and optimize the same
InfoNCE-style softmax loss we used for N-pair above. The temperature `τ` controls how sharply the
softmax focuses on the hardest negatives.

**References:**
1. van den Oord, A., Li, Y., & Vinyals, O. (2018). *Representation Learning with Contrastive
   Predictive Coding*. arXiv:1807.03748.
2. Chen, T., Kornblith, S., Norouzi, M., & Hinton, G. (2020). *A Simple Framework for Contrastive
   Learning of Visual Representations* (SimCLR). ICML.